# 04 — Modélisation prédictive du churn

## Ce que je vais faire ici
On a notre table de 4 518 clients avec 12 features et une cible CHURN. 
On va entraîner **3 modèles ML** pour apprendre à prédire le churn, les 
comparer rigoureusement, et choisir le meilleur.

## Plan :
1. **Préparation** : chargement, imputation des NaN, encodage
2. **Split** train/test stratifié (80/20)
3. **3 modèles** entraînés :
   - Logistic Regression (baseline simple et interprétable)
   - Random Forest (puissant, gère les non-linéarités)
   - XGBoost (standard de l'industrie ML tabulaire)
4. **Évaluation** : ROC-AUC, PR-AUC, F1, matrice de confusion
5. **Validation croisée** sur le meilleur modèle
6. **Feature importance** : qu'est-ce qui prédit le churn ?
7. **Threshold tuning** : combien de churneurs détectés à différents seuils ?
8. **Sortie Power BI** : scores de churn par client

## Pourquoi cette étape est cruciale
La segmentation (étape 3) dit "qui sont mes clients". La modélisation dit 
"quels clients vont churner et qu'est-ce que je peux y faire". C'est la 
partie qui rend ton projet **actionnable** pour un client e-commerce.

## 1. Imports et chargement

On charge la table enrichie de l'étape 3. On garde toutes les features 
sauf `segment_rfm` et `kmeans_cluster` qui ne doivent pas servir au modèle 
(elles sont dérivées des features RFM, donc redondantes).

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

# XGBoost
from xgboost import XGBClassifier

# Configuration
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")
np.random.seed(42)

# Chargement
DATA_PROCESSED = Path("../data/processed")
df = pd.read_csv(DATA_PROCESSED / "customer_features_segmented.csv")

print(f"Données chargées : {len(df):,} clients × {df.shape[1]} colonnes")
print(f"\nTaux de churn : {df['churn'].mean()*100:.2f}%")

Données chargées : 4,518 clients × 19 colonnes

Taux de churn : 48.92%


## 2. Préparation des features

On sépare X (les features) et y (la cible). On exclut :
- `customer_id` : identifiant, pas une feature
- `churn` : la cible elle-même
- `n_orders_future` : information du futur (data leakage)
- `segment_rfm`, `RFM_score`, `R_score`, `F_score`, `M_score`, `kmeans_cluster` : 
  dérivés des features de base

On garde nos **12 features originales**.

In [3]:
# Colonnes à exclure
cols_to_drop = [
    'customer_id', 'churn', 'n_orders_future',
    'segment_rfm', 'RFM_score', 
    'R_score', 'F_score', 'M_score', 
    'kmeans_cluster'
]

X = df.drop(columns=cols_to_drop)
y = df['churn']

print(f"✅ Features utilisées ({X.shape[1]}) :")
for col in X.columns:
    print(f"  - {col}")

print(f"\nTaille de X : {X.shape}")
print(f"Distribution de y : {y.value_counts().to_dict()}")

✅ Features utilisées (10) :
  - recency
  - frequency
  - monetary
  - avg_basket
  - std_basket
  - n_distinct_products
  - tenure_days
  - avg_interpurchase_days
  - n_cancellations
  - cancellation_rate

Taille de X : (4518, 10)
Distribution de y : {0: 2308, 1: 2210}


## 3. Imputation des NaN

On a 31% de NaN sur `std_basket` et `avg_interpurchase_days` (les 
mono-acheteurs).

**Stratégie** :
- `std_basket = 0` → cohérent (un client avec 1 commande a variance 0)
- `avg_interpurchase_days = max observé` → traduit "intervalle infini"

XGBoost gère nativement les NaN, mais Logistic Regression et Random Forest 
non. Donc on impute pour avoir un dataset propre pour les 3 modèles.

In [4]:
# Imputation
X['std_basket'] = X['std_basket'].fillna(0)
X['avg_interpurchase_days'] = X['avg_interpurchase_days'].fillna(X['avg_interpurchase_days'].max())

print(f"✅ NaN restants : {X.isna().sum().sum()}")
print(f"   (devrait être 0)")

✅ NaN restants : 0
   (devrait être 0)
